Upload do dataset, primeiros passos e análise de qualidade dos dados.

In [402]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/raw/transactional-sample.csv")

In [403]:
df.head()

,transaction_id,merchant_id,user_id,card_number,transaction_date,transaction_amount,device_id,has_cbk,transaction_datetime,transaction_hour,transaction_day_of_week,transaction_date_only,transaction_period,amount_bin
0,21320398,29744,97051,434505******9116,2019-12-01T23:16:32.812632,374.56,285475.0,False,2019-12-01 23:16:32.812632,23,Sunday,2019-12-01,Noite,"(306.854, 571.128]"
1,21320399,92895,2708,444456******4210,2019-12-01T22:45:37.873639,734.87,497105.0,True,2019-12-01 22:45:37.873639,22,Sunday,2019-12-01,Noite,"(571.128, 1197.854]"
2,21320400,47759,14777,425850******7024,2019-12-01T22:22:43.021495,760.36,NaN,False,2019-12-01 22:22:43.021495,22,Sunday,2019-12-01,Noite,"(571.128, 1197.854]"
3,21320401,68657,69758,464296******3991,2019-12-01T21:59:19.797129,2556.13,NaN,True,2019-12-01 21:59:19.797129,21,Sunday,2019-12-01,Noite,"(1197.854, 4097.21]"
4,21320402,54075,64367,650487******6116,2019-12-01T21:30:53.347051,55.36,860232.0,False,2019-12-01 21:30:53.347051,21,Sunday,2019-12-01,Noite,"(1.219, 170.566]"


In [404]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3199 entries, 0 to 3198
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   transaction_id           3199 non-null   int64         
 1   merchant_id              3199 non-null   int64         
 2   user_id                  3199 non-null   int64         
 3   card_number              3199 non-null   str           
 4   transaction_date         3199 non-null   str           
 5   transaction_amount       3199 non-null   float64       
 6   device_id                2369 non-null   float64       
 7   has_cbk                  3199 non-null   bool          
 8   transaction_datetime     3199 non-null   datetime64[us]
 9   transaction_hour         3199 non-null   int32         
 10  transaction_day_of_week  3199 non-null   str           
 11  transaction_date_only    3199 non-null   object        
 12  transaction_period       3199 non-null   str 

In [405]:
print("Colunas:")
for column in df.columns:
    print(f"- {column}")

Colunas:
- transaction_id
- merchant_id
- user_id
- card_number
- transaction_date
- transaction_amount
- device_id
- has_cbk
- transaction_datetime
- transaction_hour
- transaction_day_of_week
- transaction_date_only
- transaction_period
- amount_bin


In [406]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
})

missing.sort_values("missing_count", ascending=False)

,missing_count,missing_pct
device_id,830,25.945608
transaction_id,0,0.000000
user_id,0,0.000000
card_number,0,0.000000
transaction_date,0,0.000000
merchant_id,0,0.000000
transaction_amount,0,0.000000
has_cbk,0,0.000000
transaction_datetime,0,0.000000
transaction_hour,0,0.000000


In [407]:
print(f"Linhas duplicadas completas: {df.duplicated().sum():,}")

Linhas duplicadas completas: 0


In [408]:
id_columns = [
    "transaction_id",
    "merchant_id",
    "user_id",
    "card_number",
    "device_id",
]

for column in id_columns:
    print(
        f"{column}: "
        f"{df[column].nunique(dropna=True):,} únicos | "
        f"{df[column].duplicated().sum():,} duplicados"
    )

transaction_id: 3,199 únicos | 0 duplicados
merchant_id: 1,756 únicos | 1,443 duplicados
user_id: 2,704 únicos | 495 duplicados
card_number: 2,925 únicos | 274 duplicados
device_id: 1,996 únicos | 1,202 duplicados


In [409]:
print(df["has_cbk"].value_counts(dropna=False))
print()
print(df["has_cbk"].value_counts(normalize=True, dropna=False).mul(100).round(2))

has_cbk
False    2808
True      391
Name: count, dtype: int64

has_cbk
False    87.78
True     12.22
Name: proportion, dtype: float64


In [410]:
df["transaction_amount"].describe()

count    3199.000000
mean      767.812904
std       889.095904
min         1.220000
25%       205.235000
50%       415.940000
75%       981.680000
max      4097.210000
Name: transaction_amount, dtype: float64

In [411]:
print("Valores nulos:", df["transaction_amount"].isna().sum())
print("Valores <= 0:", (df["transaction_amount"] <= 0).sum())
print("Valores negativos:", (df["transaction_amount"] < 0).sum())

Valores nulos: 0
Valores <= 0: 0
Valores negativos: 0


In [412]:
transaction_dates = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

print("Datas inválidas:", transaction_dates.isna().sum())
print("Data mínima:", transaction_dates.min())
print("Data máxima:", transaction_dates.max())

Datas inválidas: 0
Data mínima: 2019-11-01 01:27:15.811098
Data máxima: 2019-12-01 23:16:32.812632


In [413]:
print(transaction_dates.describe())

count                          3199
mean     2019-11-22 12:47:02.242350
min      2019-11-01 01:27:15.811098
25%      2019-11-18 18:35:57.557813
50%      2019-11-23 13:50:58.758108
75%      2019-11-28 21:51:06.055557
max      2019-12-01 23:16:32.812632
Name: transaction_date, dtype: object


In [414]:
print("Devices únicos:", df["device_id"].nunique())
print("Transactions sem device:", df["device_id"].isna().sum())
print("Transactions com device:", df["device_id"].notna().sum())

Devices únicos: 1996
Transactions sem device: 830
Transactions com device: 2369


In [415]:
df["device_id"].value_counts(dropna=False).head(10)

device_id
NaN         830
563499.0     22
342890.0     19
101848.0     17
438940.0     14
547440.0     13
274282.0      8
223682.0      7
589318.0      7
542535.0      7
Name: count, dtype: int64

In [416]:
print("Valores de has_cbk:")
print(df["has_cbk"].unique())

print("\nTipos dos identificadores:")
for column in ["transaction_id", "merchant_id", "user_id"]:
    print(f"{column}: {df[column].dtype}")

print("\nTransaction IDs:")
print(f"Mínimo: {df['transaction_id'].min()}")
print(f"Máximo: {df['transaction_id'].max()}")
print(f"Únicos: {df['transaction_id'].nunique()}")
print(f"Linhas: {len(df)}")

Valores de has_cbk:
[False  True]

Tipos dos identificadores:
transaction_id: int64
merchant_id: int64
user_id: int64

Transaction IDs:
Mínimo: 21320398
Máximo: 21323596
Únicos: 3199
Linhas: 3199


## Consistency of relationships between entities

Beyond the uniqueness of identifiers, it is necessary to evaluate how users, cards, devices, and merchants relate to one another.

In the context of transaction risk, these relationships can represent significant signals regarding behavior, resource sharing, or potential fraud patterns.

At this stage, no transformations will be applied to the data. The goal is simply to characterize the existing relationships.

In [417]:
entity_counts = {
    "Transações": df["transaction_id"].nunique(),
    "Usuários": df["user_id"].nunique(),
    "Merchants": df["merchant_id"].nunique(),
    "Cartões": df["card_number"].nunique(),
    "Dispositivos": df["device_id"].nunique(),
}

pd.Series(entity_counts)

Transações      3199
Usuários        2704
Merchants       1756
Cartões         2925
Dispositivos    1996
dtype: int64

In [418]:
print("Transações por usuário:")
print(df["user_id"].value_counts().describe())

print("\nTransações por merchant:")
print(df["merchant_id"].value_counts().describe())

print("\nTransações por cartão:")
print(df["card_number"].value_counts().describe())

print("\nTransações por dispositivo:")
print(df["device_id"].value_counts().describe())

Transações por usuário:
count    2704.000000
mean        1.183062
std         1.052521
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        31.000000
Name: count, dtype: float64

Transações por merchant:
count    1756.000000
mean        1.821754
std         2.676606
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        73.000000
Name: count, dtype: float64

Transações por cartão:
count    2925.000000
mean        1.093675
std         0.432255
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        10.000000
Name: count, dtype: float64

Transações por dispositivo:
count    1996.000000
mean        1.186874
std         1.008062
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        22.000000
Name: count, dtype: float64


In [419]:
card_user = (
    df.groupby("card_number")["user_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Cartões associados a mais de um usuário:")
print((card_user > 1).sum())

print("\nMaior número de usuários por cartão:")
print(card_user.max())

Cartões associados a mais de um usuário:
31

Maior número de usuários por cartão:
4


In [420]:
device_user = (
    df.dropna(subset=["device_id"])
    .groupby("device_id")["user_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Dispositivos associados a mais de um usuário:")
print((device_user > 1).sum())

print("\nMaior número de usuários por dispositivo:")
print(device_user.max())

Dispositivos associados a mais de um usuário:
0

Maior número de usuários por dispositivo:
1


In [421]:
user_device = (
    df.dropna(subset=["device_id"])
    .groupby("user_id")["device_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Usuários associados a mais de um dispositivo:")
print((user_device > 1).sum())

print("\nMaior número de dispositivos por usuário:")
print(user_device.max())

Usuários associados a mais de um dispositivo:
24

Maior número de dispositivos por usuário:
4


## Transaction frequency by entity and chargeback

After evaluating the relationships between entities, this step compares transaction frequency between records with and without chargebacks.

The objective is to identify descriptive patterns that may subsequently contribute to risk analysis.

The results will not be interpreted in isolation as evidence of fraud.

In [422]:
transaction_frequency = pd.DataFrame({
    "transactions_per_user": df.groupby("user_id")["transaction_id"].transform("count"),
    "transactions_per_card": df.groupby("card_number")["transaction_id"].transform("count"),
    "transactions_per_merchant": df.groupby("merchant_id")["transaction_id"].transform("count"),
    "transactions_per_device": (
        df.groupby("device_id")["transaction_id"].transform("count")
    ),
    "has_cbk": df["has_cbk"],
})

transaction_frequency.groupby("has_cbk").describe().T

has_cbk                                False       True 
transactions_per_user     count  2808.000000  391.000000
                          mean      1.387464    7.373402
                          std       1.881267    8.237420
                          min       1.000000    1.000000
                          25%       1.000000    2.000000
                          50%       1.000000    4.000000
                          75%       1.000000    9.000000
                          max      31.000000   31.000000
transactions_per_card     count  2808.000000  391.000000
                          mean      1.148860    2.094629
                          std       0.568680    1.719793
                          min       1.000000    1.000000
                          25%       1.000000    1.000000
                          50%       1.000000    1.000000
                          75%       1.000000    3.000000
                          max       7.000000   10.000000
transactions_per_merchant count  2808.000000  391.000000
                          mean      5.264601    9.253197
                          std      11.804554    7.543707
                          min       1.000000    1.000000
                          25%       1.000000    4.000000
                          50%       2.000000    6.000000
                          75%       4.000000   12.000000
                          max      73.000000   30.000000
transactions_per_device   count  2045.000000  324.000000
                          mean      1.338386    6.487654
                          std       1.524079    6.423388
                          min       1.000000    1.000000
                          25%       1.000000    2.000000
                          50%       1.000000    4.000000
                          75%       1.000000    7.000000
                          max      22.000000   22.000000

In [423]:
frequency_summary = (
    transaction_frequency
    .groupby("has_cbk")
    .agg(
        user_frequency_mean=("transactions_per_user", "mean"),
        user_frequency_max=("transactions_per_user", "max"),
        card_frequency_mean=("transactions_per_card", "mean"),
        card_frequency_max=("transactions_per_card", "max"),
        merchant_frequency_mean=("transactions_per_merchant", "mean"),
        merchant_frequency_max=("transactions_per_merchant", "max"),
        device_frequency_mean=("transactions_per_device", "mean"),
        device_frequency_max=("transactions_per_device", "max"),
    )
)

frequency_summary

,user_frequency_mean,user_frequency_max,card_frequency_mean,card_frequency_max,merchant_frequency_mean,merchant_frequency_max,device_frequency_mean,device_frequency_max
has_cbk,,,,,,,,
False,1.387464,31,1.148860,7,5.264601,73,1.338386,22.0
True,7.373402,31,2.094629,10,9.253197,30,6.487654,22.0


## Transaction value and chargebacks

In this step, we compare the distribution of transaction values ​​between operations with and without chargebacks.

The objective is to determine whether there are descriptive differences in transaction value between the two groups.

This analysis is exploratory and does not imply a causal relationship between value and the occurrence of chargebacks.

In [424]:
value_by_cbk = (
    df.groupby("has_cbk")["transaction_amount"]
    .describe()
    .T
)

value_by_cbk

has_cbk,False,True
count,2808.000000,391.000000
mean,672.324380,1453.571918
std,797.463853,1169.491346
min,1.220000,2.890000
25%,191.285000,565.580000
50%,360.315000,999.470000
75%,812.577500,2140.680000
max,4091.830000,4097.210000


In [425]:
value_summary = (
    df.groupby("has_cbk")["transaction_amount"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
)

value_summary

,count,mean,median,std,min,max
has_cbk,,,,,,
False,2808,672.324380,360.315,797.463853,1.22,4091.83
True,391,1453.571918,999.470,1169.491346,2.89,4097.21


In [426]:
value_quantiles = (
    df.groupby("has_cbk")["transaction_amount"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .unstack()
)

value_quantiles

,0.25,0.50,0.75,0.90,0.95,0.99
has_cbk,,,,,,
False,191.285,360.315,812.5775,1787.817,2488.708,3856.4869
True,565.580,999.470,2140.6800,3557.900,3987.325,4078.6960


In [427]:
value_summary["mean_to_median_ratio"] = (
    value_summary["mean"] / value_summary["median"]
)

value_summary

,count,mean,median,std,min,max,mean_to_median_ratio
has_cbk,,,,,,,
False,2808,672.324380,360.315,797.463853,1.22,4091.83,1.865935
True,391,1453.571918,999.470,1169.491346,2.89,4097.21,1.454343


### Preliminary findings

Transactions involving chargebacks exhibit substantially higher transaction values ​​than those without chargebacks.

The median transaction value was R$ 999.47 for transactions with chargebacks, compared to R$ 360.32 for those without. At the third quartile, the difference remains significant: R$ 2,140.68 versus R$ 812.58.

This pattern suggests that `transaction_amount` may be a relevant variable for characterizing transaction risk.

However, this difference is descriptive and should not be interpreted in isolation as causal evidence of chargebacks.

## Temporal patterns

The temporal dimension will be analyzed to determine whether the occurrence of chargebacks is concentrated during specific times or periods.

The original `transaction_date` column will be preserved. The temporal variables used in this analysis will be derived exclusively for analytical purposes.

In [428]:
df["transaction_datetime"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce",
)

df["transaction_hour"] = df["transaction_datetime"].dt.hour
df["transaction_day_of_week"] = df["transaction_datetime"].dt.day_name()
df["transaction_date_only"] = df["transaction_datetime"].dt.date

In [429]:
hour_cbk = (
    df.groupby("transaction_hour")["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
        chargeback_rate="mean",
    )
)

hour_cbk["chargeback_rate"] = (
    hour_cbk["chargeback_rate"] * 100
).round(2)

hour_cbk

,transactions,chargebacks,chargeback_rate
transaction_hour,,,
0,133,24,18.05
1,111,14,12.61
2,61,18,29.51
3,30,4,13.33
4,7,0,0.00
5,4,0,0.00
6,2,1,50.00
8,3,0,0.00
9,7,0,0.00


In [430]:
weekday_cbk = (
    df.groupby("transaction_day_of_week")["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
        chargeback_rate="mean",
    )
)

weekday_cbk["chargeback_rate"] = (
    weekday_cbk["chargeback_rate"] * 100
).round(2)

weekday_cbk

,transactions,chargebacks,chargeback_rate
transaction_day_of_week,,,
Friday,805,120,14.91
Monday,284,20,7.04
Saturday,756,99,13.10
Sunday,392,41,10.46
Thursday,549,61,11.11
Tuesday,256,16,6.25
Wednesday,157,34,21.66


## Time of Day

To reduce the granularity of the hourly analysis and facilitate interpretation, transactions will be grouped into time-of-day periods.

These periods are defined solely for exploratory purposes and do not represent a risk classification.

In [431]:
def classify_period(hour):
    if 0 <= hour < 6:
        return "Madrugada"
    if 6 <= hour < 12:
        return "Manhã"
    if 12 <= hour < 18:
        return "Tarde"
    return "Noite"


df["transaction_period"] = df["transaction_hour"].apply(classify_period)

In [432]:
period_cbk = (
    df.groupby("transaction_period")["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
        chargeback_rate="mean",
    )
)

period_cbk["chargeback_rate"] = (
    period_cbk["chargeback_rate"] * 100
).round(2)

period_cbk

,transactions,chargebacks,chargeback_rate
transaction_period,,,
Madrugada,346,60,17.34
Manhã,134,6,4.48
Noite,1378,210,15.24
Tarde,1341,115,8.58


In [433]:
period_order = ["Madrugada", "Manhã", "Tarde", "Noite"]

period_cbk = period_cbk.reindex(period_order)

period_cbk

,transactions,chargebacks,chargeback_rate
transaction_period,,,
Madrugada,346,60,17.34
Manhã,134,6,4.48
Tarde,1341,115,8.58
Noite,1378,210,15.24


In [434]:
entity_analysis = {}

for column in ["user_id", "merchant_id", "card_number", "device_id"]:
    grouped = (
        df.groupby(column, dropna=False)
        .agg(
            transactions=("transaction_id", "count"),
            chargebacks=("has_cbk", "sum"),
        )
    )

    grouped["chargeback_rate"] = (
        grouped["chargebacks"] / grouped["transactions"] * 100
    ).round(2)

    entity_analysis[column] = grouped

entity_analysis

{'user_id':          transactions  chargebacks  chargeback_rate
 user_id                                            
 6                   1            0              0.0
 7                   1            0              0.0
 8                   1            0              0.0
 19                  1            0              0.0
 132                 1            0              0.0
 ...               ...          ...              ...
 99863               1            0              0.0
 99875               1            0              0.0
 99876               1            1            100.0
 99948               1            0              0.0
 99974               1            0              0.0
 
 [2704 rows x 3 columns],
 'merchant_id':              transactions  chargebacks  chargeback_rate
 merchant_id                                            
 16                      2            0              0.0
 54                      1            0              0.0
 65                      1   

In [435]:
entity_analysis["user_id"].sort_values(
    ["chargebacks", "transactions"],
    ascending=False
).head(20)

,transactions,chargebacks,chargeback_rate
user_id,,,
11750,31,25,80.65
91637,22,19,86.36
79054,17,15,88.24
96025,14,13,92.86
78262,13,12,92.31
75710,10,10,100.00
7725,7,7,100.00
17929,6,6,100.00
21768,6,6,100.00


In [436]:
entity_analysis["merchant_id"].sort_values(
    ["chargebacks", "transactions"],
    ascending=False
).head(20)

,transactions,chargebacks,chargeback_rate
merchant_id,,,
17275,30,22,73.33
4705,22,19,86.36
1308,15,15,100.00
53041,19,14,73.68
77130,15,13,86.67
91972,14,11,78.57
44927,11,11,100.00
73271,10,10,100.00
55854,11,9,81.82


In [437]:
entity_analysis["card_number"].sort_values(
    ["chargebacks", "transactions"],
    ascending=False
).head(20)

,transactions,chargebacks,chargeback_rate
card_number,,,
554482******7640,10,10,100.0
530034******3859,6,6,100.0
651653******2256,5,5,100.0
441030******2146,4,4,100.0
459080******2870,4,4,100.0
498406******7104,4,4,100.0
530034******8258,4,4,100.0
552289******8870,4,4,100.0
550209******3098,4,3,75.0


In [438]:
entity_analysis["device_id"].sort_values(
    ["chargebacks", "transactions"],
    ascending=False
).head(20)

,transactions,chargebacks,chargeback_rate
device_id,,,
NaN,830,67,8.07
563499.0,22,19,86.36
342890.0,19,15,78.95
101848.0,17,15,88.24
438940.0,14,13,92.86
547440.0,13,12,92.31
542535.0,7,6,85.71
357277.0,6,6,100.00
960729.0,6,6,100.00


In [439]:
# Dispositivos associados a múltiplos cartões
device_card_links = (
    df.dropna(subset=["device_id"])
    .groupby("device_id")["card_number"]
    .nunique()
    .sort_values(ascending=False)
)

print("Dispositivos associados a mais de um cartão:")
print((device_card_links > 1).sum())

print("\nMáximo de cartões por dispositivo:")
print(device_card_links.max())

Dispositivos associados a mais de um cartão:
94

Máximo de cartões por dispositivo:
22


In [440]:
# Cartões associados a múltiplos dispositivos
card_device_links = (
    df.dropna(subset=["device_id"])
    .groupby("card_number")["device_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Cartões associados a mais de um dispositivo:")
print((card_device_links > 1).sum())

print("\nMáximo de dispositivos por cartão:")
print(card_device_links.max())

Cartões associados a mais de um dispositivo:
28

Máximo de dispositivos por cartão:
4


In [441]:
# Usuários associados a múltiplos cartões
user_card_links = (
    df.groupby("user_id")["card_number"]
    .nunique()
    .sort_values(ascending=False)
)

print("Usuários associados a mais de um cartão:")
print((user_card_links > 1).sum())

print("\nMáximo de cartões por usuário:")
print(user_card_links.max())

Usuários associados a mais de um cartão:
114

Máximo de cartões por usuário:
31


Key finding

The most interesting case is:

1 device associated with up to 22 cards.

This warrants investigation because a device used with many cards could indicate legitimate sharing, a shared environment, or potentially suspicious behavior. We should not conclude fraud based solely on this indicator.

Similarly:

1 user associated with up to 31 cards.

This behavior differs significantly from that of the average user and warrants treatment as a potential risk signal; however, we need to examine the distribution before setting any thresholds.

In [442]:
device_card_risk = (
    df.dropna(subset=["device_id"])
    .groupby("device_id")
    .agg(
        transactions=("transaction_id", "count"),
        cards=("card_number", "nunique"),
        users=("user_id", "nunique"),
        chargebacks=("has_cbk", "sum"),
    )
)

device_card_risk["chargeback_rate"] = (
    device_card_risk["chargebacks"]
    / device_card_risk["transactions"]
    * 100
).round(2)

device_card_risk.sort_values(
    ["cards", "chargebacks"],
    ascending=False
).head(20)

,transactions,cards,users,chargebacks,chargeback_rate
device_id,,,,,
563499.0,22,22,1,19,86.36
342890.0,19,19,1,15,78.95
101848.0,17,15,1,15,88.24
438940.0,14,10,1,13,92.86
547440.0,13,10,1,12,92.31
542535.0,7,7,1,6,85.71
223682.0,7,7,1,4,57.14
960729.0,6,6,1,6,100.00
262327.0,6,6,1,4,66.67


In [443]:
user_card_risk = (
    df.groupby("user_id")
    .agg(
        transactions=("transaction_id", "count"),
        cards=("card_number", "nunique"),
        devices=("device_id", "nunique"),
        chargebacks=("has_cbk", "sum"),
    )
)

user_card_risk["chargeback_rate"] = (
    user_card_risk["chargebacks"]
    / user_card_risk["transactions"]
    * 100
).round(2)

user_card_risk.sort_values(
    ["cards", "chargebacks"],
    ascending=False
).head(20)

,transactions,cards,devices,chargebacks,chargeback_rate
user_id,,,,,
11750,31,31,4,25,80.65
91637,22,22,1,19,86.36
79054,17,15,1,15,88.24
96025,14,10,1,13,92.86
78262,13,10,1,12,92.31
7695,7,7,1,4,57.14
17929,6,6,1,6,100.00
67519,6,6,1,4,66.67
34548,6,6,2,0,0.00


In [444]:
total_chargebacks = df["has_cbk"].sum()
total_transactions = len(df)

print(f"Chargebacks: {total_chargebacks:,}")
print(f"Transações: {total_transactions:,}")
print(
    f"Taxa global de chargeback: "
    f"{total_chargebacks / total_transactions * 100:.2f}%"
)

Chargebacks: 391
Transações: 3,199
Taxa global de chargeback: 12.22%


In [445]:
top_users = (
    entity_analysis["user_id"]
    .sort_values(["chargebacks", "transactions"], ascending=False)
    .head(10)
)

top_merchants = (
    entity_analysis["merchant_id"]
    .sort_values(["chargebacks", "transactions"], ascending=False)
    .head(10)
)

print("Top 10 usuários por chargebacks:")
display(top_users)

print("Top 10 merchants por chargebacks:")
display(top_merchants)

Top 10 usuários por chargebacks:


,transactions,chargebacks,chargeback_rate
user_id,,,
11750,31,25,80.65
91637,22,19,86.36
79054,17,15,88.24
96025,14,13,92.86
78262,13,12,92.31
75710,10,10,100.00
7725,7,7,100.00
17929,6,6,100.00
21768,6,6,100.00


Top 10 merchants por chargebacks:


,transactions,chargebacks,chargeback_rate
merchant_id,,,
17275,30,22,73.33
4705,22,19,86.36
1308,15,15,100.00
53041,19,14,73.68
77130,15,13,86.67
91972,14,11,78.57
44927,11,11,100.00
73271,10,10,100.00
55854,11,9,81.82


The next section should answer a more important question:

Which transaction characteristics are associated with a higher probability of a chargeback?

In [446]:
df["amount_bin"] = pd.qcut(
    df["transaction_amount"],
    q=5,
    duplicates="drop"
)

amount_analysis = (
    df.groupby("amount_bin", observed=True)
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
    )
)

amount_analysis["chargeback_rate"] = (
    amount_analysis["chargebacks"]
    / amount_analysis["transactions"]
    * 100
).round(2)

amount_analysis

,transactions,chargebacks,chargeback_rate
amount_bin,,,
"(1.219, 170.566]",640,19,2.97
"(170.566, 306.854]",640,19,2.97
"(306.854, 571.128]",639,65,10.17
"(571.128, 1197.854]",640,127,19.84
"(1197.854, 4097.21]",640,161,25.16


In [447]:
amount_period_analysis = (
    df.groupby(
        ["amount_bin", "transaction_period"],
        observed=True
    )
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
    )
)

amount_period_analysis["chargeback_rate"] = (
    amount_period_analysis["chargebacks"]
    / amount_period_analysis["transactions"]
    * 100
).round(2)

amount_period_analysis

transactions  chargebacks  \
amount_bin          transaction_period                              
(1.219, 170.566]    Madrugada                     88            3   
                    Manhã                         32            0   
                    Noite                        287           13   
                    Tarde                        233            3   
(170.566, 306.854]  Madrugada                     55            3   
                    Manhã                         20            1   
                    Noite                        266           10   
                    Tarde                        299            5   
(306.854, 571.128]  Madrugada                     78           10   
                    Manhã                         27            0   
                    Noite                        268           35   
                    Tarde                        266           20   
(571.128, 1197.854] Madrugada                     72           24   
                    Manhã                         25            2   
                    Noite                        261           55   
                    Tarde                        282           46   
(1197.854, 4097.21] Madrugada                     53           20   
                    Manhã                         30            3   
                    Noite                        296           97   
                    Tarde                        261           41   

                                        chargeback_rate  
amount_bin          transaction_period                   
(1.219, 170.566]    Madrugada                      3.41  
                    Manhã                          0.00  
                    Noite                          4.53  
                    Tarde                          1.29  
(170.566, 306.854]  Madrugada                      5.45  
                    Manhã                          5.00  
                    Noite                          3.76  
                    Tarde                          1.67  
(306.854, 571.128]  Madrugada                     12.82  
                    Manhã                          0.00  
                    Noite                         13.06  
                    Tarde                          7.52  
(571.128, 1197.854] Madrugada                     33.33  
                    Manhã                          8.00  
                    Noite                         21.07  
                    Tarde                         16.31  
(1197.854, 4097.21] Madrugada                     37.74  
                    Manhã                         10.00  
                    Noite                         32.77  
                    Tarde                         15.71

In [448]:
device_presence_analysis = (
    df.assign(
        device_status=df["device_id"].isna().map({
            True: "Sem dispositivo",
            False: "Com dispositivo"
        })
    )
    .groupby("device_status")
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
    )
)

device_presence_analysis["chargeback_rate"] = (
    device_presence_analysis["chargebacks"]
    / device_presence_analysis["transactions"]
    * 100
).round(2)

device_presence_analysis

,transactions,chargebacks,chargeback_rate
device_status,,,
Com dispositivo,2369,324,13.68
Sem dispositivo,830,67,8.07


We already have enough information to start investigating combinations of signals.

In [449]:
amount_device_analysis = (
    df.assign(
        device_status=df["device_id"].isna().map({
            True: "Sem dispositivo",
            False: "Com dispositivo"
        })
    )
    .groupby(
        ["amount_bin", "device_status"],
        observed=True
    )
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
    )
)

amount_device_analysis["chargeback_rate"] = (
    amount_device_analysis["chargebacks"]
    / amount_device_analysis["transactions"]
    * 100
).round(2)

amount_device_analysis

transactions  chargebacks  \
amount_bin          device_status                                
(1.219, 170.566]    Com dispositivo           480           15   
                    Sem dispositivo           160            4   
(170.566, 306.854]  Com dispositivo           482           14   
                    Sem dispositivo           158            5   
(306.854, 571.128]  Com dispositivo           478           57   
                    Sem dispositivo           161            8   
(571.128, 1197.854] Com dispositivo           486          111   
                    Sem dispositivo           154           16   
(1197.854, 4097.21] Com dispositivo           443          127   
                    Sem dispositivo           197           34   

                                     chargeback_rate  
amount_bin          device_status                     
(1.219, 170.566]    Com dispositivo             3.12  
                    Sem dispositivo             2.50  
(170.566, 306.854]  Com dispositivo             2.90  
                    Sem dispositivo             3.16  
(306.854, 571.128]  Com dispositivo            11.92  
                    Sem dispositivo             4.97  
(571.128, 1197.854] Com dispositivo            22.84  
                    Sem dispositivo            10.39  
(1197.854, 4097.21] Com dispositivo            28.67  
                    Sem dispositivo            17.26

In [450]:
user_risk = (
    df.groupby("user_id")
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
        total_amount=("transaction_amount", "sum"),
        avg_amount=("transaction_amount", "mean"),
    )
)

user_risk["chargeback_rate"] = (
    user_risk["chargebacks"]
    / user_risk["transactions"]
    * 100
).round(2)

user_risk.sort_values(
    ["chargebacks", "transactions"],
    ascending=False
).head(20)

,transactions,chargebacks,total_amount,avg_amount,chargeback_rate
user_id,,,,,
11750,31,25,17816.26,574.718065,80.65
91637,22,19,17335.51,787.977727,86.36
79054,17,15,35497.93,2088.113529,88.24
96025,14,13,30200.22,2157.158571,92.86
78262,13,12,39195.13,3015.010000,92.31
75710,10,10,5613.81,561.381000,100.00
7725,7,7,10822.94,1546.134286,100.00
17929,6,6,21840.69,3640.115000,100.00
21768,6,6,5152.61,858.768333,100.00


In [451]:
user_frequency_analysis = (
    df.groupby("user_id")
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
    )
)

user_frequency_analysis["frequency_bin"] = pd.cut(
    user_frequency_analysis["transactions"],
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=[
        "1 transação",
        "2 transações",
        "3–5 transações",
        "6–10 transações",
        "Mais de 10 transações",
    ],
)

frequency_analysis = (
    user_frequency_analysis
    .groupby("frequency_bin", observed=True)
    .agg(
        users=("transactions", "count"),
        transactions=("transactions", "sum"),
        chargebacks=("chargebacks", "sum"),
    )
)

frequency_analysis["chargeback_rate"] = (
    frequency_analysis["chargebacks"]
    / frequency_analysis["transactions"]
    * 100
).round(2)

frequency_analysis

,users,transactions,chargebacks,chargeback_rate
frequency_bin,,,,
1 transação,2469,2469,64,2.59
2 transações,155,310,60,19.35
3–5 transações,60,217,126,58.06
6–10 transações,15,106,57,53.77
Mais de 10 transações,5,97,84,86.60


The key insight

There is a strong association between user recurrence and chargebacks.

A user with only one transaction has a chargeback rate of 2.59%.

When a user has more than 10 transactions, that figure reaches 86.60%.

This represents a difference of approximately 33x between the two extremes.

And more importantly: it is not simply because there are more transactions. The chargeback rate itself increases drastically as recurrence increases.

In [452]:
user_frequency_amount = (
    df.groupby("user_id")
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
        avg_amount=("transaction_amount", "mean"),
    )
)

user_frequency_amount["frequency_bin"] = pd.cut(
    user_frequency_amount["transactions"],
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=[
        "1 transação",
        "2 transações",
        "3–5 transações",
        "6–10 transações",
        "Mais de 10 transações",
    ],
)

frequency_amount_analysis = (
    user_frequency_amount
    .groupby("frequency_bin", observed=True)
    .agg(
        users=("transactions", "count"),
        avg_transactions=("transactions", "mean"),
        avg_amount=("avg_amount", "mean"),
        total_chargebacks=("chargebacks", "sum"),
        users_with_chargeback=(
            "chargebacks",
            lambda x: (x > 0).sum()
        ),
    )
)

frequency_amount_analysis

,users,avg_transactions,avg_amount,total_chargebacks,users_with_chargeback
frequency_bin,,,,,
1 transação,2469,1.000000,685.247007,64,64
2 transações,155,2.000000,892.073742,60,34
3–5 transações,60,3.616667,1028.274703,126,39
6–10 transações,15,7.066667,1272.968691,57,11
Mais de 10 transações,5,19.400000,1724.595579,84,5


The group with more than 10 transactions is particularly interesting:

only 5 users;
an average of 19.4 transactions per user;
highest average value: R$ 1,724.60;
84 chargebacks;
all 5 users have at least one chargeback.

This reinforces the hypothesis that high transaction frequency combined with higher values ​​is associated with a higher concentration of chargebacks.

Next step: creating features

Let's start without a score.

First, we will create a transaction-level analytical table containing the history available prior to the transaction itself.

This is important because we want to avoid data leakage.

For example, we cannot use:

"how many chargebacks this user had across the entire dataset"

to evaluate a past transaction, because we would be using information from the future.

This precaution is crucial for this case study.

Next cell

First, let's create a copy sorted by time:

In [453]:
total_chargebacks = df["has_cbk"].sum()
total_transactions = len(df)

print(f"Chargebacks: {total_chargebacks:,}")
print(f"Transações: {total_transactions:,}")
print(
    f"Taxa global de chargeback: "
    f"{total_chargebacks / total_transactions * 100:.2f}%"
)

Chargebacks: 391
Transações: 3,199
Taxa global de chargeback: 12.22%


In [454]:
df_risk = df.copy()

df_risk["transaction_date"] = pd.to_datetime(
    df_risk["transaction_date"]
)

df_risk = (
    df_risk
    .sort_values("transaction_date")
    .reset_index(drop=True)
)

df_risk.head()

,transaction_id,merchant_id,user_id,card_number,transaction_date,transaction_amount,device_id,has_cbk,transaction_datetime,transaction_hour,transaction_day_of_week,transaction_date_only,transaction_period,amount_bin
0,21323596,17348,8,650487******9884,2019-11-01 01:27:15.811098,2416.70,NaN,False,2019-11-01 01:27:15.811098,1,Friday,2019-11-01,Madrugada,"(1197.854, 4097.21]"
1,21323595,35930,7,544315******7773,2019-11-01 01:29:45.799767,359.68,NaN,False,2019-11-01 01:29:45.799767,1,Friday,2019-11-01,Madrugada,"(306.854, 571.128]"
2,21323594,57997,84486,522688******9874,2019-11-01 10:23:50.555604,1.55,NaN,False,2019-11-01 10:23:50.555604,10,Friday,2019-11-01,Manhã,"(1.219, 170.566]"
3,21323593,9603,59275,528052******3611,2019-11-01 11:47:02.404963,1403.67,NaN,False,2019-11-01 11:47:02.404963,11,Friday,2019-11-01,Manhã,"(1197.854, 4097.21]"
4,21323592,50493,49581,650486******4139,2019-11-01 13:05:34.054967,744.15,NaN,False,2019-11-01 13:05:34.054967,13,Friday,2019-11-01,Tarde,"(571.128, 1197.854]"


In [455]:
df_risk["user_previous_transactions"] = (
    df_risk.groupby("user_id").cumcount()
)

df_risk["card_previous_transactions"] = (
    df_risk.groupby("card_number").cumcount()
)

df_risk["merchant_previous_transactions"] = (
    df_risk.groupby("merchant_id").cumcount()
)

df_risk["device_previous_transactions"] = (
    df_risk.groupby("device_id").cumcount()
)

In [456]:
df_risk[
    [
        "transaction_id",
        "transaction_date",
        "user_id",
        "card_number",
        "merchant_id",
        "device_id",
        "user_previous_transactions",
        "card_previous_transactions",
        "merchant_previous_transactions",
        "device_previous_transactions",
    ]
].head(20)

,transaction_id,transaction_date,user_id,card_number,merchant_id,device_id,user_previous_transactions,card_previous_transactions,merchant_previous_transactions,device_previous_transactions
0,21323596,2019-11-01 01:27:15.811098,8,650487******9884,17348,NaN,0,0,0,NaN
1,21323595,2019-11-01 01:29:45.799767,7,544315******7773,35930,NaN,0,0,0,NaN
2,21323594,2019-11-01 10:23:50.555604,84486,522688******9874,57997,NaN,0,0,0,NaN
3,21323593,2019-11-01 11:47:02.404963,59275,528052******3611,9603,NaN,0,0,0,NaN
4,21323592,2019-11-01 13:05:34.054967,49581,650486******4139,50493,NaN,0,0,0,NaN
5,21323591,2019-11-01 14:30:36.417602,19248,476331******8121,13371,NaN,0,0,0,NaN
6,21323590,2019-11-01 14:33:11.280461,38415,544731******7009,15140,NaN,0,0,0,NaN
7,21323589,2019-11-01 16:20:00.178559,84944,530033******1098,13371,NaN,0,0,1,NaN
8,21323588,2019-11-01 16:50:32.916048,93741,530034******1987,50493,NaN,0,0,1,NaN
9,21323587,2019-11-01 17:52:57.071163,19,525496******3638,82477,NaN,0,0,0,NaN


In [457]:
history_analysis = pd.DataFrame({
    "user_previous_transactions": df_risk["user_previous_transactions"],
    "card_previous_transactions": df_risk["card_previous_transactions"],
    "merchant_previous_transactions": df_risk["merchant_previous_transactions"],
    "device_previous_transactions": df_risk["device_previous_transactions"],
    "has_cbk": df_risk["has_cbk"],
})

history_analysis.groupby("has_cbk").agg(
    user_history_mean=("user_previous_transactions", "mean"),
    user_history_max=("user_previous_transactions", "max"),
    card_history_mean=("card_previous_transactions", "mean"),
    card_history_max=("card_previous_transactions", "max"),
    merchant_history_mean=("merchant_previous_transactions", "mean"),
    merchant_history_max=("merchant_previous_transactions", "max"),
    device_history_mean=("device_previous_transactions", "mean"),
    device_history_max=("device_previous_transactions", "max"),
)

,user_history_mean,user_history_max,card_history_mean,card_history_max,merchant_history_mean,merchant_history_max,device_history_mean,device_history_max
has_cbk,,,,,,,,
False,0.174858,26,0.073362,6,2.119302,72,0.150611,17.0
True,3.322251,30,0.554987,9,4.219949,29,2.861111,21.0


In [458]:
user_history_risk = (
    df_risk.assign(
        user_has_history=df_risk["user_previous_transactions"] > 0
    )
    .groupby("user_has_history")["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
        chargeback_rate="mean",
    )
)

user_history_risk["chargeback_rate"] = (
    user_history_risk["chargeback_rate"] * 100
).round(2)

user_history_risk

,transactions,chargebacks,chargeback_rate
user_has_history,,,
False,2704,131,4.84
True,495,260,52.53


In [459]:
entity_history_results = {}

for entity in ["card", "merchant", "device"]:
    column = f"{entity}_previous_transactions"

    result = (
        df_risk.assign(
            has_history=df_risk[column].fillna(0) > 0
        )
        .groupby("has_history")["has_cbk"]
        .agg(
            transactions="count",
            chargebacks="sum",
            chargeback_rate="mean",
        )
    )

    result["chargeback_rate"] = (
        result["chargeback_rate"] * 100
    ).round(2)

    entity_history_results[entity] = result

entity_history_results

{'card':              transactions  chargebacks  chargeback_rate
 has_history                                            
 False                2925          268             9.16
 True                  274          123            44.89,
 'merchant':              transactions  chargebacks  chargeback_rate
 has_history                                            
 False                1756           84             4.78
 True                 1443          307            21.28,
 'device':              transactions  chargebacks  chargeback_rate
 has_history                                            
 False                2826          179             6.33
 True                  373          212            56.84}

This demonstrates something very important:

The existence of a prior history is strongly associated with chargeback risk across all the entities analyzed.

The strongest signals are:

Device: 56.84% vs. 6.33%
User: 52.53% vs. 4.84%
Card: 44.89% vs. 9.16%
Merchant: 21.28% vs. 4.78%
However, there is an important detail.

We should not simply interpret this as:

"If there is a history, then it is fraud."

This is because we are looking at an observational dataset, and `has_cbk` is the known outcome of the transaction.

What we can professionally state is:

Transaction history is a strong discriminative signal for risk in this dataset.

This is precisely the type of conclusion that matters in a risk analysis case study.

In [460]:
df_risk["user_previous_chargebacks"] = (
    df_risk.groupby("user_id")["has_cbk"]
    .cumsum()
    .shift(fill_value=0)
)

df_risk["card_previous_chargebacks"] = (
    df_risk.groupby("card_number")["has_cbk"]
    .cumsum()
    .shift(fill_value=0)
)

df_risk["merchant_previous_chargebacks"] = (
    df_risk.groupby("merchant_id")["has_cbk"]
    .cumsum()
    .shift(fill_value=0)
)

df_risk["device_previous_chargebacks"] = (
    df_risk.groupby("device_id")["has_cbk"]
    .cumsum()
    .shift(fill_value=0)
)

In [461]:
df_risk["user_previous_chargebacks"] = (
    df_risk.groupby("user_id")["has_cbk"]
    .transform(lambda x: x.astype(int).cumsum().shift(fill_value=0))
)

df_risk["card_previous_chargebacks"] = (
    df_risk.groupby("card_number")["has_cbk"]
    .transform(lambda x: x.astype(int).cumsum().shift(fill_value=0))
)

df_risk["merchant_previous_chargebacks"] = (
    df_risk.groupby("merchant_id")["has_cbk"]
    .transform(lambda x: x.astype(int).cumsum().shift(fill_value=0))
)

df_risk["device_previous_chargebacks"] = (
    df_risk.groupby("device_id")["has_cbk"]
    .transform(lambda x: x.astype(int).cumsum().shift(fill_value=0))
)

In [462]:
df_risk[
    [
        "user_id",
        "has_cbk",
        "user_previous_chargebacks",
        "card_previous_chargebacks",
        "merchant_previous_chargebacks",
        "device_previous_chargebacks",
    ]
].head(20)

,user_id,has_cbk,user_previous_chargebacks,card_previous_chargebacks,merchant_previous_chargebacks,device_previous_chargebacks
0,8,False,0,0,0,NaN
1,7,False,0,0,0,NaN
2,84486,False,0,0,0,NaN
3,59275,False,0,0,0,NaN
4,49581,False,0,0,0,NaN
5,19248,False,0,0,0,NaN
6,38415,False,0,0,0,NaN
7,84944,False,0,0,0,NaN
8,93741,False,0,0,0,NaN
9,19,False,0,0,0,NaN


In [463]:
df_risk[
    [
        "user_previous_chargebacks",
        "card_previous_chargebacks",
        "merchant_previous_chargebacks",
        "device_previous_chargebacks",
    ]
].describe()

,user_previous_chargebacks,card_previous_chargebacks,merchant_previous_chargebacks,device_previous_chargebacks
count,3199.000000,3199.000000,3199.000000,2369.000000
mean,0.372929,0.065958,0.492967,0.359223
std,1.877261,0.428460,1.975743,1.628673
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000
max,24.000000,9.000000,21.000000,18.000000


In [464]:
previous_cbk_analysis = {}

for entity in ["user", "card", "merchant", "device"]:
    column = f"{entity}_previous_chargebacks"

    result = (
        df_risk.assign(
            has_previous_chargeback=df_risk[column].fillna(0) > 0
        )
        .groupby("has_previous_chargeback")["has_cbk"]
        .agg(
            transactions="count",
            chargebacks="sum",
            chargeback_rate="mean",
        )
    )

    result["chargeback_rate"] = (
        result["chargeback_rate"] * 100
    ).round(2)

    previous_cbk_analysis[entity] = result

previous_cbk_analysis

{'user':                          transactions  chargebacks  chargeback_rate
 has_previous_chargeback                                            
 False                            2934          153             5.21
 True                              265          238            89.81,
 'card':                          transactions  chargebacks  chargeback_rate
 has_previous_chargeback                                            
 False                            3079          274              8.9
 True                              120          117             97.5,
 'merchant':                          transactions  chargebacks  chargeback_rate
 has_previous_chargeback                                            
 False                            2853          118             4.14
 True                              346          273            78.90,
 'device':                          transactions  chargebacks  chargeback_rate
 has_previous_chargeback                                      

In [465]:
df_risk["high_amount"] = (
    df_risk["transaction_amount"] >
    df_risk["transaction_amount"].quantile(0.75)
)

df_risk["user_previous_cbk_flag"] = (
    df_risk["user_previous_chargebacks"] > 0
)

combined_risk = (
    df_risk.groupby(
        ["high_amount", "user_previous_cbk_flag"]
    )["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
        chargeback_rate="mean",
    )
)

combined_risk["chargeback_rate"] = (
    combined_risk["chargeback_rate"] * 100
).round(2)

combined_risk

transactions  chargebacks  chargeback_rate
high_amount user_previous_cbk_flag                                            
False       False                           2257           65             2.88
            True                             142          123            86.62
True        False                            677           88            13.00
            True                             123          115            93.50

In [466]:
df_risk["high_amount_flag"] = (
    df_risk["transaction_amount"] > df_risk["transaction_amount"].quantile(0.75)
).astype(int)

df_risk["user_previous_cbk_flag"] = (
    df_risk["user_previous_chargebacks"] > 0
).astype(int)

df_risk["card_previous_cbk_flag"] = (
    df_risk["card_previous_chargebacks"] > 0
).astype(int)

df_risk["merchant_previous_cbk_flag"] = (
    df_risk["merchant_previous_chargebacks"] > 0
).astype(int)

df_risk["device_previous_cbk_flag"] = (
    df_risk["device_previous_chargebacks"] > 0
).astype(int)

In [467]:
df_risk["risk_signal_count"] = (
    df_risk["high_amount_flag"]
    + df_risk["user_previous_cbk_flag"]
    + df_risk["card_previous_cbk_flag"]
    + df_risk["merchant_previous_cbk_flag"]
    + df_risk["device_previous_cbk_flag"]
)

In [468]:
risk_signal_analysis = (
    df_risk.groupby("risk_signal_count")
    .agg(
        transactions=("transaction_id", "count"),
        chargebacks=("has_cbk", "sum"),
    )
)

risk_signal_analysis["chargeback_rate"] = (
    risk_signal_analysis["chargebacks"]
    / risk_signal_analysis["transactions"]
    * 100
).round(2)

risk_signal_analysis

,transactions,chargebacks,chargeback_rate
risk_signal_count,,,
0,2206,48,2.18
1,670,63,9.40
2,76,55,72.37
3,101,87,86.14
4,106,98,92.45
5,40,40,100.00


In [469]:
risk_flags = {
    "user_previous_cbk": df_risk["user_previous_chargebacks"] > 0,
    "card_previous_cbk": df_risk["card_previous_chargebacks"] > 0,
    "merchant_previous_cbk": df_risk["merchant_previous_chargebacks"] > 0,
    "device_previous_cbk": df_risk["device_previous_chargebacks"] > 0,
    "high_amount": df_risk["transaction_amount"] > df_risk["transaction_amount"].median(),
}

risk_flag_analysis = {}

for flag_name, flag in risk_flags.items():
    analysis = (
        df_risk.assign(risk_flag=flag)
        .groupby("risk_flag")["has_cbk"]
        .agg(
            transactions="count",
            chargebacks="sum",
            chargeback_rate="mean",
        )
    )

    analysis["chargeback_rate"] = (
        analysis["chargeback_rate"] * 100
    ).round(2)

    risk_flag_analysis[flag_name] = analysis

risk_flag_analysis

{'user_previous_cbk':            transactions  chargebacks  chargeback_rate
 risk_flag                                            
 False              2934          153             5.21
 True                265          238            89.81,
 'card_previous_cbk':            transactions  chargebacks  chargeback_rate
 risk_flag                                            
 False              3079          274              8.9
 True                120          117             97.5,
 'merchant_previous_cbk':            transactions  chargebacks  chargeback_rate
 risk_flag                                            
 False              2853          118             4.14
 True                346          273            78.90,
 'device_previous_cbk':            transactions  chargebacks  chargeback_rate
 risk_flag                                            
 False              2981          196             6.57
 True                218          195            89.45,
 'high_amount':           

In [470]:
df_risk["user_previous_cbk_flag"] = (
    df_risk["user_previous_chargebacks"] > 0
).astype(int)

df_risk["card_previous_cbk_flag"] = (
    df_risk["card_previous_chargebacks"] > 0
).astype(int)

df_risk["merchant_previous_cbk_flag"] = (
    df_risk["merchant_previous_chargebacks"] > 0
).astype(int)

df_risk["device_previous_cbk_flag"] = (
    df_risk["device_previous_chargebacks"] > 0
).astype(int)

df_risk["high_amount_flag"] = (
    df_risk["transaction_amount"] > df_risk["transaction_amount"].median()
).astype(int)

df_risk["risk_score"] = (
    df_risk["user_previous_cbk_flag"]
    + df_risk["card_previous_cbk_flag"]
    + df_risk["merchant_previous_cbk_flag"]
    + df_risk["device_previous_cbk_flag"]
    + df_risk["high_amount_flag"]
)

In [471]:
risk_score_analysis = (
    df_risk.groupby("risk_score")["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
        chargeback_rate="mean",
    )
)

risk_score_analysis["chargeback_rate"] = (
    risk_score_analysis["chargeback_rate"] * 100
).round(2)

risk_score_analysis

,transactions,chargebacks,chargeback_rate
risk_score,,,
0,1547,25,1.62
1,1298,72,5.55
2,95,58,61.05
3,42,37,88.10
4,151,134,88.74
5,66,65,98.48


In [472]:
risk_threshold_analysis = (
    df_risk.assign(
        high_risk=df_risk["risk_score"] >= 2
    )
    .groupby("high_risk")["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
    )
)

risk_threshold_analysis["transaction_pct"] = (
    risk_threshold_analysis["transactions"]
    / len(df_risk)
    * 100
).round(2)

risk_threshold_analysis["chargeback_capture_pct"] = (
    risk_threshold_analysis["chargebacks"]
    / df_risk["has_cbk"].sum()
    * 100
).round(2)

risk_threshold_analysis

,transactions,chargebacks,transaction_pct,chargeback_capture_pct
high_risk,,,,
False,2845,97,88.93,24.81
True,354,294,11.07,75.19


In [473]:
threshold_results = []

total_chargebacks = df_risk["has_cbk"].sum()

for threshold in range(1, 6):
    predicted_high_risk = df_risk["risk_score"] >= threshold

    transactions = predicted_high_risk.sum()
    chargebacks = df_risk.loc[predicted_high_risk, "has_cbk"].sum()

    precision = (
        chargebacks / transactions * 100
        if transactions > 0
        else 0
    )

    recall = (
        chargebacks / total_chargebacks * 100
        if total_chargebacks > 0
        else 0
    )

    transaction_pct = (
        transactions / len(df_risk) * 100
    )

    threshold_results.append(
        {
            "threshold": threshold,
            "transactions": transactions,
            "transaction_pct": round(transaction_pct, 2),
            "chargebacks_captured": chargebacks,
            "chargeback_capture_pct": round(recall, 2),
            "precision_pct": round(precision, 2),
        }
    )

threshold_analysis = pd.DataFrame(threshold_results)

threshold_analysis

,threshold,transactions,transaction_pct,chargebacks_captured,chargeback_capture_pct,precision_pct
0,1,1652,51.64,366,93.61,22.15
1,2,354,11.07,294,75.19,83.05
2,3,259,8.10,236,60.36,91.12
3,4,217,6.78,199,50.90,91.71
4,5,66,2.06,65,16.62,98.48


In [474]:
risk_signal_summary = (
    df_risk[df_risk["risk_score"] >= 2]
    [
        [
            "risk_score",
            "user_previous_cbk_flag",
            "card_previous_cbk_flag",
            "merchant_previous_cbk_flag",
            "device_previous_cbk_flag",
            "high_amount_flag",
            "has_cbk",
        ]
    ]
)

risk_signal_summary.head(20)

,risk_score,user_previous_cbk_flag,card_previous_cbk_flag,merchant_previous_cbk_flag,device_previous_cbk_flag,high_amount_flag,has_cbk
20,3,1,0,1,0,1,True
38,2,0,0,1,0,1,False
41,4,1,1,1,0,1,True
42,3,1,1,1,0,0,True
57,4,1,1,1,0,1,True
58,4,1,1,1,0,1,True
59,4,1,1,1,0,1,True
63,2,0,0,1,0,1,True
89,4,1,1,1,0,1,True
90,4,1,1,1,0,1,True


In [475]:
risk_signal_combinations = (
    df_risk[df_risk["risk_score"] >= 2]
    .groupby(
        [
            "user_previous_cbk_flag",
            "card_previous_cbk_flag",
            "merchant_previous_cbk_flag",
            "device_previous_cbk_flag",
            "high_amount_flag",
        ]
    )
    ["has_cbk"]
    .agg(
        transactions="count",
        chargebacks="sum",
    )
    .reset_index()
)

risk_signal_combinations["chargeback_rate"] = (
    risk_signal_combinations["chargebacks"]
    / risk_signal_combinations["transactions"]
    * 100
).round(2)

risk_signal_combinations.sort_values(
    ["chargebacks", "transactions"],
    ascending=False,
).head(20)

,user_previous_cbk_flag,card_previous_cbk_flag,merchant_previous_cbk_flag,device_previous_cbk_flag,high_amount_flag,transactions,chargebacks,chargeback_rate
10,1,0,1,1,1,113,96,84.96
17,1,1,1,1,1,66,65,98.48
0,0,0,1,0,1,82,51,62.20
15,1,1,1,0,1,18,18,100.00
8,1,0,1,0,1,15,15,100.00
16,1,1,1,1,0,14,14,100.00
6,1,0,0,1,1,10,8,80.00
14,1,1,1,0,0,7,7,100.00
13,1,1,0,1,1,6,6,100.00
9,1,0,1,1,0,6,3,50.00


In [476]:
threshold = 2

y_true = df_risk["has_cbk"].astype(bool)
y_pred = df_risk["risk_score"] >= threshold

true_positive = (y_true & y_pred).sum()
false_positive = (~y_true & y_pred).sum()
false_negative = (y_true & ~y_pred).sum()
true_negative = (~y_true & ~y_pred).sum()

precision = true_positive / (true_positive + false_positive)
recall = true_positive / (true_positive + false_negative)
specificity = true_negative / (true_negative + false_positive)

final_metrics = pd.Series(
    {
        "threshold": threshold,
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision_pct": precision * 100,
        "recall_pct": recall * 100,
        "specificity_pct": specificity * 100,
        "review_rate_pct": y_pred.mean() * 100,
    }
)

final_metrics.round(2)

threshold             2.00
true_positive       294.00
false_positive       60.00
false_negative       97.00
true_negative      2748.00
precision_pct        83.05
recall_pct           75.19
specificity_pct      97.86
review_rate_pct      11.07
dtype: float64

In [477]:
print("=== RISK SCORING — RESULTADO FINAL ===")
print()

print(f"Transações analisadas: {len(df_risk):,}")
print(f"Chargebacks: {df_risk['has_cbk'].sum():,}")
print(
    f"Taxa global de chargeback: "
    f"{df_risk['has_cbk'].mean() * 100:.2f}%"
)

print()
print(f"Threshold selecionado: risk_score >= 2")
print(f"Transações para revisão: {y_pred.sum():,}")
print(f"Percentual para revisão: {y_pred.mean() * 100:.2f}%")
print(f"Chargebacks capturados: {true_positive:,}")
print(f"Chargebacks não capturados: {false_negative:,}")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"Specificity: {specificity * 100:.2f}%")

=== RISK SCORING — RESULTADO FINAL ===

Transações analisadas: 3,199
Chargebacks: 391
Taxa global de chargeback: 12.22%

Threshold selecionado: risk_score >= 2
Transações para revisão: 354
Percentual para revisão: 11.07%
Chargebacks capturados: 294
Chargebacks não capturados: 97
Precision: 83.05%
Recall: 75.19%
Specificity: 97.86%


In [478]:
financial_risk_analysis = (
    df_risk.assign(
        high_risk=df_risk["risk_score"] >= 2
    )
    .groupby("high_risk")
    .agg(
        transactions=("transaction_id", "count"),
        total_amount=("transaction_amount", "sum"),
        chargebacks=("has_cbk", "sum"),
        chargeback_amount=(
            "transaction_amount",
            lambda x: x[df_risk.loc[x.index, "has_cbk"]].sum(),
        ),
    )
)

financial_risk_analysis["transaction_pct"] = (
    financial_risk_analysis["transactions"]
    / len(df_risk)
    * 100
).round(2)

financial_risk_analysis["amount_pct"] = (
    financial_risk_analysis["total_amount"]
    / df_risk["transaction_amount"].sum()
    * 100
).round(2)

financial_risk_analysis["chargeback_amount_pct"] = (
    financial_risk_analysis["chargeback_amount"]
    / df_risk.loc[df_risk["has_cbk"], "transaction_amount"].sum()
    * 100
).round(2)

financial_risk_analysis.round(2)

,transactions,total_amount,chargebacks,chargeback_amount,transaction_pct,amount_pct,chargeback_amount_pct
high_risk,,,,,,,
False,2845,1951841.43,97,131540.30,88.93,79.46,23.14
True,354,504392.05,294,436806.32,11.07,20.54,76.86


In [479]:
confusion_matrix = pd.DataFrame(
    {
        "Predito: Alto Risco": [294, 60],
        "Predito: Baixo Risco": [97, 2748],
    },
    index=["Real: Chargeback", "Real: Não Chargeback"],
)

confusion_matrix

,Predito: Alto Risco,Predito: Baixo Risco
Real: Chargeback,294,97
Real: Não Chargeback,60,2748


In [480]:
tp = 294
fn = 97
fp = 60
tn = 2748

precision = tp / (tp + fp)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
review_rate = (tp + fp) / (tp + fn + fp + tn)

print("=== MÉTRICAS FINAIS ===")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"Specificity: {specificity * 100:.2f}%")
print(f"Review rate: {review_rate * 100:.2f}%")

=== MÉTRICAS FINAIS ===
Precision: 83.05%
Recall: 75.19%
Specificity: 97.86%
Review rate: 11.07%


In [481]:
print("=== RISK SCORING — RESULTADO FINAL ===")

print(f"Transações analisadas: {tp + fn + fp + tn:,}")
print(f"Chargebacks: {tp + fn:,}")
print(f"Threshold selecionado: risk_score >= 2")
print(f"Transações para revisão: {tp + fp:,}")
print(f"Percentual para revisão: {review_rate * 100:.2f}%")
print(f"Chargebacks capturados: {tp:,}")
print(f"Chargebacks não capturados: {fn:,}")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"Specificity: {specificity * 100:.2f}%")

=== RISK SCORING — RESULTADO FINAL ===
Transações analisadas: 3,199
Chargebacks: 391
Threshold selecionado: risk_score >= 2
Transações para revisão: 354
Percentual para revisão: 11.07%
Chargebacks capturados: 294
Chargebacks não capturados: 97
Precision: 83.05%
Recall: 75.19%
Specificity: 97.86%


##  Conclusions and Risk Recommendations

The analysis identified clear behavioral, transactional, and historical patterns associated with chargeback risk.

### Key findings

- The overall chargeback rate is 12.22% across 3,199 transactions.
- Chargeback risk increases substantially with transaction frequency.
- Higher transaction amounts are strongly associated with higher chargeback rates.
- Previous chargeback history is one of the strongest risk signals.
- Transactions associated with entities that previously experienced chargebacks show substantially higher chargeback rates.
- The combination of multiple risk signals produces a strong concentration of chargebacks.

### Risk scoring

An interpretable risk score was created using five binary signals:

1. Previous user chargeback
2. Previous card chargeback
3. Previous merchant chargeback
4. Previous device chargeback
5. High transaction amount

The selected operational threshold was:

`risk_score >= 2`

At this threshold:

- 354 transactions are selected for review.
- Only 11.07% of all transactions require review.
- 294 of 391 chargebacks are captured.
- Recall is 75.19%.
- Precision is 83.05%.
- Specificity is 97.86%.

### Financial impact

The high-risk segment represents:

- 11.07% of transactions.
- 20.54% of total transaction value.
- 75.19% of all chargebacks.
- 76.86% of the total chargeback amount.

This indicates that the risk score concentrates a disproportionate share of financial risk into a relatively small review population.

### Business recommendation

The risk score can be used as a transaction prioritization mechanism rather than as an automatic blocking rule.

Transactions with `risk_score >= 2` should be prioritized for manual review or additional verification.

Lower-risk transactions can follow the standard processing flow, reducing unnecessary operational effort while maintaining substantial chargeback coverage.

### Limitations

The analysis is based on historical transactional data and an interpretable rule-based scoring approach. The results should therefore be considered a prioritization framework rather than a production fraud detection model.

Before production deployment, the approach should be validated using temporal holdout data, monitored for changes in chargeback behavior, and evaluated against operational review capacity and false-positive costs.